In [ ]:
# Force uninstall the broken version
!pip uninstall -y tensorflow keras tensorboard

# Clear the cache to free up space
!pip cache purge

# Reinstall 
!pip install --user tensorflow

In [17]:
import pandas as pd

# Load the dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
column_names = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
                'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
                'hours-per-week', 'native-country', 'income']
data = pd.read_csv(url, header=None, names=column_names, na_values=" ?", skipinitialspace=True)

#dataset does not have headers so we manually create them 

# Drop rows with missing values
data.dropna(inplace=True)
data.shape

(32561, 15)

In [18]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [19]:
# preprocess
# Encode target ">50K" -> 1, "<=50K" > 0
data['income'] = (data['income'].astype(str).str.strip() == '>50K').astype(int)

# One-hot encode categorical columns
cat_cols = ['workclass', 'education', 'marital-status', 'occupation',
            'relationship', 'race', 'sex', 'native-country']
df_final = pd.get_dummies(data, columns=cat_cols)

#separate features and target
X = df_final.drop('income', axis=1)
y = df_final['income']

# Train/test split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print(f"Training samples : {X_train.shape[0]}")
print(f"Test samples     : {X_test.shape[0]}")
print(f"Features         : {X_train.shape[1]}")

#X_train is a 2D array

Training samples : 26048
Test samples     : 6513
Features         : 108


In [20]:
#building neural network
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Initialize the model
model = Sequential()
# model that does forward pass, backpropagation, gradient descent. Allows layers to be stacked in a sequence.
# you could build these layers on your own without sequential as well - 
# that would be helpful in learning the model design process, but it does require writing a lot more code.

# Hidden Layer 1: 64 neurons, ReLU activation
# input_shape matches the number of columns in our processed X_train
model.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
#input_shape -> This tells the network how many "Input Layer" features x1, x2...to expect. 
#Since we one-hot encoded the Adult dataset, this number is quite high.

# Hidden Layer 2: 32 neurons, ReLU activation
model.add(Dense(32, activation='relu'))

# Output Layer: 1 neuron, Sigmoid activation (for 0 to 1 probability)
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(optimizer='adam', # A popular version of Mini-Batch Gradient Descent
              loss='binary_crossentropy', # Loss function for binary classification
              metrics=['accuracy'])

/network/rit/home/sg186634/.local/lib/python3.9/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [21]:
# Train the model

#model.fit command starts the iterative process of forward pass, loss calculation, backprogragation, and weight updates
history = model.fit(X_train, y_train, 
                    epochs=20, 
                    batch_size=32, 
                    validation_split=0.2, # Uses 20% of train data to check for overfitting
                    verbose=1)
#common values for batch size = 32,64,128
#Smaller batches -> noisier updates but can generalise better
#Larger batches -> smoother updates but needs more memory
#32 or 64 is a safe default for tabular data

# Evaluate on the unseen test set
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"\nTest Accuracy: {test_acc:.4f}")

Epoch 1/20
652/652 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8209 - loss: 0.3938 - val_accuracy: 0.8484 - val_loss: 0.3264
Epoch 2/20
652/652 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8508 - loss: 0.3122 - val_accuracy: 0.8530 - val_loss: 0.3186
Epoch 3/20
652/652 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8546 - loss: 0.3089 - val_accuracy: 0.8534 - val_loss: 0.3221
Epoch 4/20
652/652 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8588 - loss: 0.3059 - val_accuracy: 0.8495 - val_loss: 0.3211
Epoch 5/20
652/652 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8630 - loss: 0.2909 - val_accuracy: 0.8516 - val_loss: 0.3183
Epoch 6/20
652/652 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8632 - loss: 0.2916 - val_accuracy: 0.8541 - val_loss: 0.3196
Epoch 7/20
652/652 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8667 - loss: 0.2849 - val_accuracy: 0.8532 - val_loss: 0.3226
Epoch 8/20
652/652 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8683 - loss: 0.2831 - val_accuracy: 0.